# Step 00 — Migrate OLD 12:00-Snapshot ERA5 Files to Backup
**Project:** ENSO-BSISO Self-Supervised Learning  
**Author:** Jiayi (jh9141@nyu.edu)

**Run this notebook ONCE, on Colab, BEFORE re-running the download notebooks**
(`01`, `01b`, `01c`, `12`) after the switch from the single 12:00 UTC snapshot to
the new **daily-average** download logic (Session 33).

## Why this is needed
Every download cell starts with a skip-if-exists guard:
```python
if os.path.exists(out_file):
    print('[SKIP] ... already exists'); continue
```
The new daily-average code writes to the **exact same filenames** as the old snapshot code.
So if the old files are still present, every chunk is skipped and the new logic never runs —
you would silently keep the snapshot data with no error. This notebook moves the old files
into a `_snapshot12z_backup/` subfolder so the skip guard lets the new code re-download.

## Safety
- **Moves, never deletes** — your old data is preserved under `_snapshot12z_backup/`.
- **Idempotent** — if a file is already backed up, the current file is left untouched, so it is
  safe to re-run even after the new daily files exist (it will NOT move fresh daily data).
- Backup lives in a subfolder; the preprocessing/verify globs use a non-recursive `os.listdir`,
  so backed-up files are invisible to them.

## After running this
1. Re-run `01`, `01b`, `01c`, `12` → they re-download with daily-average logic.
2. Re-run preprocessing (`03`, `13`, `13b`, `09`) and downstream training/NSV.
3. Once the new downloads look right, delete `_snapshot12z_backup/` to reclaim Drive space.

---

## Cell 1 — Mount Google Drive

In [ ]:
from google.colab import drive
drive.mount('/content/drive')
print('Google Drive mounted.')

## Cell 2 — Move Old Snapshot Files to `_snapshot12z_backup/`

In [ ]:
import os, shutil

BSISO_RAW = '/content/drive/MyDrive/BSISO_SSL_Project/data/raw'
MJO_RAW   = '/content/drive/MyDrive/BSISO_SSL_Project/MJO/data/raw'

# Output-file prefixes produced by the four download notebooks.
patterns = {
    BSISO_RAW: ('u850_v850_July_', 'OLR_July_', 'u850_v850_MJJAS_', 'OLR_MJJAS_', 'precip_MJJAS_'),
    MJO_RAW:   ('u850_u200_', 'OLR_MJO_'),
}

total_moved = 0
for raw_dir, prefixes in patterns.items():
    if not os.path.isdir(raw_dir):
        print(f'[skip] {raw_dir} not found')
        continue
    backup = os.path.join(raw_dir, '_snapshot12z_backup')
    os.makedirs(backup, exist_ok=True)
    moved = 0
    for f in sorted(os.listdir(raw_dir)):
        if not (f.endswith('.nc') and f.startswith(prefixes)):
            continue
        dst = os.path.join(backup, f)
        if os.path.exists(dst):
            # Already migrated -> leave the current file in place. This protects freshly
            # re-downloaded daily-average files if this notebook is run a second time.
            print(f'[already backed up] {f}  (left current file untouched)')
            continue
        shutil.move(os.path.join(raw_dir, f), dst)
        moved += 1
    total_moved += moved
    print(f'{raw_dir}: moved {moved} file(s) -> _snapshot12z_backup/')

print(f'\nDone. {total_moved} file(s) moved to backup.')
if total_moved == 0:
    print('Nothing moved -> either already migrated, or no old snapshot files were present.')

## Cell 3 — Verify: `raw/` is now clear of old files, backup holds them

In [ ]:
import os

for raw_dir in [BSISO_RAW, MJO_RAW]:
    if not os.path.isdir(raw_dir):
        continue
    nc_now    = sorted(f for f in os.listdir(raw_dir) if f.endswith('.nc'))
    backup    = os.path.join(raw_dir, '_snapshot12z_backup')
    nc_backup = sorted(f for f in os.listdir(backup) if f.endswith('.nc')) if os.path.isdir(backup) else []
    print(f'\n{raw_dir}')
    print(f'  .nc files still in raw/      : {len(nc_now)}')
    for f in nc_now:
        print(f'      {f}')
    print(f'  .nc files in backup/         : {len(nc_backup)}')

print('\nIf raw/ still lists snapshot files above, re-check the prefixes in Cell 2.')
print('Otherwise you are ready to re-run the download notebooks (01, 01b, 01c, 12).')

---
*DDCS Project | jh9141@nyu.edu*